In [1]:
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

# Project pathsx
DATA_RAW = Path("../data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

print("Setup complete. Data will be saved to:", DATA_RAW.resolve())

Setup complete. Data will be saved to: /Users/ramyashreeb/Projects/explainable-ai-grid/data/raw


In [2]:
# Quick test pull — current intensity
r = requests.get("https://api.carbonintensity.org.uk/intensity")
print(r.status_code)
print(r.json())

200
{'data': [{'from': '2026-07-06T22:00Z', 'to': '2026-07-06T22:30Z', 'intensity': {'forecast': 123, 'actual': 122, 'index': 'moderate'}}]}


In [3]:
frames = []

for year in range(2021, 2024):
    for month in range(1, 13):
        url = f"https://api.carbonintensity.org.uk/intensity/{year}-{month:02d}-01T00:00Z/{year}-{month:02d}-28T23:30Z"
        r = requests.get(url)
        if r.status_code == 200:
            data = r.json().get("data", [])
            if data:
                frames.append(pd.DataFrame(data))
        else:
            print(f"Failed: {year}-{month:02d} (status {r.status_code})")

carbon_df = pd.concat(frames, ignore_index=True)
print(f"Carbon intensity: {carbon_df.shape}")
carbon_df.head()

Carbon intensity: (48237, 3)


,from,to,intensity
0,2020-12-31T23:30Z,2021-01-01T00:00Z,"{'forecast': 190, 'actual': 184, 'index': 'mod..."
1,2021-01-01T00:00Z,2021-01-01T00:30Z,"{'forecast': 181, 'actual': 187, 'index': 'mod..."
2,2021-01-01T00:30Z,2021-01-01T01:00Z,"{'forecast': 178, 'actual': 182, 'index': 'mod..."
3,2021-01-01T01:00Z,2021-01-01T01:30Z,"{'forecast': 175, 'actual': 178, 'index': 'mod..."
4,2021-01-01T01:30Z,2021-01-01T02:00Z,"{'forecast': 174, 'actual': 171, 'index': 'mod..."


In [4]:
# Skip this cell until ENTSO_API_KEY is set in .env
ENTSO_KEY = os.getenv("ENTSO_API_KEY")

if ENTSO_KEY:
    from entsoe import EntsoePandasClient

    client = EntsoePandasClient(api_key=ENTSO_KEY)
    start = pd.Timestamp("2021-01-01", tz="Europe/London")
    end = pd.Timestamp("2024-01-01", tz="Europe/London")

    gen = client.query_generation("GB", start=start, end=end, psr_type=None)
    gen.to_parquet(DATA_RAW / "generation_GB_2021_2024.parquet")
    print(f"Generation: {gen.shape}")

    load = client.query_load("GB", start=start, end=end)
    load.to_parquet(DATA_RAW / "load_GB_2021_2024.parquet")
    print(f"Load: {load.shape}")
else:
    print("ENTSO_API_KEY not found in .env — register at transparency.entsoe.eu and add the key, then rerun this cell.")

Generation: (7890, 11)
Load: (7886, 1)


In [5]:
carbon_df = pd.concat(
    [carbon_df.drop(columns=["intensity"]), carbon_df["intensity"].apply(pd.Series)],
    axis=1
)
carbon_df["from"] = pd.to_datetime(carbon_df["from"])
carbon_df["to"] = pd.to_datetime(carbon_df["to"])

carbon_df.to_parquet(DATA_RAW / "carbon_intensity_GB_2021_2024.parquet")
print("Saved:", DATA_RAW / "carbon_intensity_GB_2021_2024.parquet")
carbon_df.head()

Saved: ../data/raw/carbon_intensity_GB_2021_2024.parquet


,from,to,forecast,actual,index
0,2020-12-31 23:30:00+00:00,2021-01-01 00:00:00+00:00,190,184.0,moderate
1,2021-01-01 00:00:00+00:00,2021-01-01 00:30:00+00:00,181,187.0,moderate
2,2021-01-01 00:30:00+00:00,2021-01-01 01:00:00+00:00,178,182.0,moderate
3,2021-01-01 01:00:00+00:00,2021-01-01 01:30:00+00:00,175,178.0,moderate
4,2021-01-01 01:30:00+00:00,2021-01-01 02:00:00+00:00,174,171.0,moderate


In [6]:
import cdsapi
from pathlib import Path

c = cdsapi.Client()

DATA_RAW = Path("../data/raw")

c.retrieve(
    'reanalysis-era5-single-levels',
    {
        'product_type': 'reanalysis',
        'variable': [
            '10m_u_component_of_wind',
            '10m_v_component_of_wind',
            '2m_temperature',
            'surface_solar_radiation_downwards',
            'total_precipitation',
        ],
        'year': ['2021', '2022', '2023'],
        'month': [f'{m:02d}' for m in range(1, 13)],
        'day': [f'{d:02d}' for d in range(1, 32)],
        'time': [f'{h:02d}:00' for h in range(0, 24)],
        'area': [61, -8, 49, 2],  # GB bounding box: N, W, S, E
        'format': 'netcdf',
    },
    str(DATA_RAW / 'era5_weather_GB_2021_2023.nc')
)

print("ERA5 download complete")

HTTPError: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-single-levels/execution
cost limits exceeded
Your request is too large, please reduce your selection.

In [7]:
import requests
import pandas as pd
from pathlib import Path
import time

DATA_RAW = Path("../data/raw")
frames = []

for year in range(2021, 2024):
    for month in range(1, 13):
        # Get settlement dates for each month
        start = f"{year}-{month:02d}-01"
        if month == 12:
            end = f"{year+1}-01-01"
        else:
            end = f"{year}-{month+1:02d}-01"
        
        url = f"https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELINST?settlementDateFrom={start}&settlementDateTo={end}&format=json"
        r = requests.get(url)
        
        if r.status_code == 200:
            data = r.json().get("data", [])
            if data:
                frames.append(pd.DataFrame(data))
                print(f"✅ {year}-{month:02d}: {len(data)} rows")
        else:
            print(f"❌ Failed: {year}-{month:02d} (status {r.status_code})")
        
        time.sleep(0.5)  # polite delay

elexon_df = pd.concat(frames, ignore_index=True)
print(f"\nElexon fuel mix total: {elexon_df.shape}")
elexon_df.to_parquet(DATA_RAW / "elexon_fuelinst_2021_2023.parquet")
print("Saved: elexon_fuelinst_2021_2023.parquet")
elexon_df.head()

❌ Failed: 2021-01 (status 400)
❌ Failed: 2021-02 (status 400)
❌ Failed: 2021-03 (status 400)
❌ Failed: 2021-04 (status 400)
❌ Failed: 2021-05 (status 400)
❌ Failed: 2021-06 (status 400)
❌ Failed: 2021-07 (status 400)
❌ Failed: 2021-08 (status 400)
❌ Failed: 2021-09 (status 400)
❌ Failed: 2021-10 (status 400)
❌ Failed: 2021-11 (status 400)
❌ Failed: 2021-12 (status 400)
❌ Failed: 2022-01 (status 400)
❌ Failed: 2022-02 (status 400)
❌ Failed: 2022-03 (status 400)
❌ Failed: 2022-04 (status 400)
❌ Failed: 2022-05 (status 400)
❌ Failed: 2022-06 (status 400)
❌ Failed: 2022-07 (status 400)
❌ Failed: 2022-08 (status 400)
❌ Failed: 2022-09 (status 400)
❌ Failed: 2022-10 (status 400)
❌ Failed: 2022-11 (status 400)
❌ Failed: 2022-12 (status 400)
❌ Failed: 2023-01 (status 400)
❌ Failed: 2023-02 (status 400)
❌ Failed: 2023-03 (status 400)
❌ Failed: 2023-04 (status 400)
❌ Failed: 2023-05 (status 400)
❌ Failed: 2023-06 (status 400)
❌ Failed: 2023-07 (status 400)
❌ Failed: 2023-08 (status 400)
❌ Failed

ValueError: No objects to concatenate

In [ ]:
import requests
import pandas as pd
from pathlib import Path
import time

DATA_RAW = Path("../data/raw")
frames = []

for year in range(2021, 2024):
    for month in range(1, 13):
        url = (
            f"https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELINST"
            f"?settlementDateFrom={year}-{month:02d}-01"
            f"&settlementDateTo={year}-{month:02d}-28"
            f"&format=json"
        )
        r = requests.get(url)
        if r.status_code == 200:
            data = r.json().get("data", [])
            if data:
                frames.append(pd.DataFrame(data))
                print(f"✅ {year}-{month:02d}: {len(data)} rows")
            else:
                print(f"⚠️ {year}-{month:02d}: empty response")
        else:
            print(f"❌ Failed: {year}-{month:02d} (status {r.status_code})")
        time.sleep(0.5)

elexon_df = pd.concat(frames, ignore_index=True)
print(f"\nElexon fuel mix total: {elexon_df.shape}")
elexon_df.to_parquet(DATA_RAW / "elexon_fuelinst_2021_2023.parquet")
print("Saved: elexon_fuelinst_2021_2023.parquet")
elexon_df.head()